# PYTORCH MASTERY — NOTEBOOK 10
## Chapters 53–58: Hugging Face + PyTorch → Large Language Models → Build a Mini Language Model → Text Generation → LLM Fine-Tuning → LoRA and PEFT

Continuing directly from Notebook 9. This is where the course reaches large language models directly.


### PYTORCH MASTERY PROGRESS
Current Chapter: 58 | Current Level: Large Language Models

- [x] Chapters 1–52 *(Notebooks 1–9)*
- [x] Hugging Face + PyTorch
- [x] Large Language Models
- [x] Build a Mini Language Model
- [x] Text Generation
- [x] LLM Fine-Tuning
- [x] LoRA and PEFT


---
# CHAPTER 53 — HUGGING FACE + PYTORCH

## 1. Concept
Hugging Face's `transformers` library sits **on top of** PyTorch (and other backends), providing pretrained models, tokenizers, and training utilities — directly analogous to how `torchvision` provides pretrained CNNs (Chapter 38-39). Understanding this relationship precisely matters: Hugging Face doesn't replace anything you've learned about PyTorch — every `transformers` model **is** a genuine `nn.Module` (Chapter 16), trainable with the exact same `.backward()`/optimizer/training-loop machinery this entire course has built.

## 2. The Core Components


In [ ]:
# Illustrative usage (requires internet access to download models/tokenizers,
# which this sandbox doesn't have -- the CODE PATTERNS are what matter here)

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

# AutoTokenizer: handles text -> token IDs (Chapter 47's full pipeline, but using the
# EXACT tokenization scheme the specific pretrained model was originally trained with)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "PyTorch and Hugging Face work together seamlessly."
encoded = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

print("Encoded output keys:", encoded.keys())
print("input_ids shape:", encoded["input_ids"].shape)
print("attention_mask shape:", encoded["attention_mask"].shape)
print("""
Note: the tokenizer AUTOMATICALLY produces BOTH input_ids (Chapter 47's token IDs)
AND attention_mask (Chapter 47's mask) in ONE call -- exactly what this course built
manually from scratch in Chapter 47, now provided as a ready-made, well-tested utility.
""")


Encoded output keys: KeysView({'input_ids': tensor([[  101,  1052, 22123,  2953,  2818,  1998, 17662,  2227,  2147,  2362,
         25180, 10895,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
input_ids shape: torch.Size([1, 14])
attention_mask shape: torch.Size([1, 14])

Note: the tokenizer AUTOMATICALLY produces BOTH input_ids (Chapter 47's token IDs)
AND attention_mask (Chapter 47's mask) in ONE call -- exactly what this course built
manually from scratch in Chapter 47, now provided as a ready-made, well-tested utility.



In [ ]:
import torch
# AutoModel: loads a PRETRAINED Transformer -- genuinely just an nn.Module (Chapter 16)
model = AutoModel.from_pretrained("bert-base-uncased")

print("model is an instance of nn.Module:", isinstance(model, torch.nn.Module))
print("model.parameters() works EXACTLY as with any model built in this course:")
total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")

# AutoModelForSequenceClassification: a PRETRAINED backbone + a classification head ALREADY attached --
# directly analogous to Chapter 38's transfer learning pattern (replace the final layer),
# except Hugging Face provides this WRAPPING for you
classifier_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
print("\nAutoModelForSequenceClassification automatically attaches a classification head --")
print("EXACTLY Chapter 38's 'replace model.fc' pattern, done FOR you for this specific task type.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model is an instance of nn.Module: True
model.parameters() works EXACTLY as with any model built in this course:
  Total parameters: 109,482,240


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



AutoModelForSequenceClassification automatically attaches a classification head --
EXACTLY Chapter 38's 'replace model.fc' pattern, done FOR you for this specific task type.


### Line-by-Line Explanation — Confirming the "It's Just PyTorch" Claim
- `isinstance(model, torch.nn.Module)` returning `True` is the single most important fact in this chapter: **every** Hugging Face model is a genuine `nn.Module`. This means `model.to(device)` (Chapter 27), `model.train()`/`model.eval()` (Chapter 23), `model.parameters()` fed to any `torch.optim` optimizer (Chapter 20), `torch.save(model.state_dict())` (Chapter 70), mixed precision (Chapter 29), and every other technique from this entire course applies **without modification**.
- `AutoTokenizer`/`AutoModel` — the `Auto*` prefix means these classes automatically determine the correct underlying tokenizer/model architecture class based on the pretrained model's name/configuration, without you needing to know in advance whether you're loading BERT, RoBERTa, GPT-2, or another architecture — a genuinely convenient abstraction over the many specific model classes the library provides.
- `AutoModelForSequenceClassification(..., num_labels=3)` — internally does exactly what Chapter 38 did manually (`model.fc = nn.Linear(...)`) — attaches an appropriately-shaped classification head on top of the pretrained backbone's output.

## 3. A Complete Fine-Tuning Loop, Using Only PyTorch Primitives You Already Know


In [ ]:
import torch
import torch.nn as nn

# The training loop is EXACTLY Chapter 23's train_one_epoch/evaluate pattern --
# genuinely nothing new to learn here beyond what this course already covered
def train_one_epoch_hf(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        # Hugging Face models conveniently compute the loss INTERNALLY when `labels` is
        # provided -- outputs.loss is ready to backward() directly, no separate
        # criterion(...) call needed (though you CAN compute your own loss from
        # outputs.logits manually if you need a custom loss function)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

print("train_one_epoch_hf() defined -- structurally IDENTICAL to Chapter 23's")
print("train_one_epoch(), with the only difference being outputs.loss is computed")
print("INTERNALLY by the Hugging Face model when labels= is passed to its forward call.")


train_one_epoch_hf() defined -- structurally IDENTICAL to Chapter 23's
train_one_epoch(), with the only difference being outputs.loss is computed
INTERNALLY by the Hugging Face model when labels= is passed to its forward call.


### Common Mistakes
- Believing Hugging Face requires learning an entirely separate framework/paradigm — it's built directly on PyTorch (or optionally other backends), and every technique from this course transfers directly.
- Not realizing `AutoModel` (bare backbone, no task head) versus `AutoModelForSequenceClassification`/`AutoModelForCausalLM`/etc. (backbone + an appropriate task-specific head already attached) are genuinely different classes for different purposes — loading the wrong one for your task requires manually attaching your own head anyway, defeating some of the convenience.
- Forgetting that Hugging Face models, when given `labels`, compute and return the loss as part of their output object (`outputs.loss`) — writing a separate, redundant loss computation when the model already provides one.

### Interview Insight
**Q: Is using Hugging Face's `transformers` library fundamentally different from working with PyTorch directly, in terms of what you need to know?**
A: No — every model `transformers` provides is a genuine PyTorch `nn.Module`, and every technique covered throughout this entire course (device placement, train/eval mode, optimizers, mixed precision, gradient management, model persistence) applies to Hugging Face models exactly as it would to any custom-built PyTorch model. What Hugging Face adds is convenience: pretrained weights for a huge range of architectures, standardized tokenizers matched to each model, and some task-specific head-attachment classes (`AutoModelForSequenceClassification`, etc.) that save you from manually replicating Chapter 38's "replace the final layer" pattern for common task types. Understanding PyTorch's fundamentals deeply, as this course builds, is exactly what makes Hugging Face genuinely easy to use effectively — rather than a separate skill to learn from scratch.

### Exercise
Sketch (in comments, given this sandbox's lack of internet access) a complete fine-tuning script for a Hugging Face sentiment classification model, using only the PyTorch primitives (Dataset, DataLoader, optimizer, training loop) already mastered in this course.

### Quiz (Chapter 53)
1. What single fact about Hugging Face models makes every PyTorch technique from this course directly applicable to them?
2. What's the difference between `AutoModel` and `AutoModelForSequenceClassification`?
3. Why doesn't a Hugging Face fine-tuning loop typically need a separately-defined `criterion`/loss function call?


---
# CHAPTER 54 — LARGE LANGUAGE MODELS

## 1. Concept
Large Language Models (LLMs) are, architecturally, **exactly the Transformer decoder** this course built the foundations for in Chapters 49-51 — scaled up dramatically in size (billions of parameters instead of millions), trained on vastly more data (trillions of tokens instead of a small custom dataset), using the same core training objective: **predicting the next token**, given everything before it.

## 2. Encoder-Only vs. Decoder-Only vs. Encoder-Decoder — the Architectural Family Tree
| Architecture | Examples | Typical use |
|---|---|---|
| **Encoder-only** | BERT | Understanding tasks: classification, extraction — processes the WHOLE input bidirectionally (every token can attend to every OTHER token, both before AND after it) |
| **Decoder-only** | GPT family, LLaMA, most modern LLMs | Generation tasks: the model predicts the NEXT token, attending only to PREVIOUS tokens (causally masked, next section) — the dominant architecture for modern general-purpose LLMs |
| **Encoder-Decoder** | T5, the original Transformer | Sequence-to-sequence tasks: translation, summarization — an encoder processes the input, a decoder generates the output, attending to the encoder's representation |

## 3. Causal (Masked) Self-Attention — the Key Architectural Difference From Chapter 49's Attention


In [ ]:
import torch
import torch.nn.functional as F

def causal_mask(seq_len):
    """Creates a MASK preventing each position from attending to FUTURE positions --
    THE defining architectural feature of decoder-only (GPT-style) models."""
    mask = torch.tril(torch.ones(seq_len, seq_len))   # torch.tril: lower-triangular matrix of 1s
    return mask


seq_len = 5
mask = causal_mask(seq_len)
print("Causal mask (1 = CAN attend, 0 = CANNOT attend):")
print(mask)
print("""
Reading this mask: row 0 (the FIRST token) can only attend to column 0 (itself) --
it has NO access to any future token. Row 2 (the THIRD token) can attend to columns
0, 1, 2 (itself and everything BEFORE it), but NOT columns 3, 4 (anything AFTER it).

THIS IS THE KEY DIFFERENCE from Chapter 49's attention: BERT-style (encoder) attention
has NO such mask -- every position can see every OTHER position, both before and after.
GPT-style (decoder) attention uses EXACTLY this triangular mask, ensuring the model
can NEVER "cheat" by looking at future tokens it's supposed to be PREDICTING --
essential for the next-token-prediction training objective to be a genuinely valid,
non-trivial task at all.
""")


Causal mask (1 = CAN attend, 0 = CANNOT attend):
tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

Reading this mask: row 0 (the FIRST token) can only attend to column 0 (itself) --
it has NO access to any future token. Row 2 (the THIRD token) can attend to columns
0, 1, 2 (itself and everything BEFORE it), but NOT columns 3, 4 (anything AFTER it).

THIS IS THE KEY DIFFERENCE from Chapter 49's attention: BERT-style (encoder) attention
has NO such mask -- every position can see every OTHER position, both before and after.
GPT-style (decoder) attention uses EXACTLY this triangular mask, ensuring the model
can NEVER "cheat" by looking at future tokens it's supposed to be PREDICTING --
essential for the next-token-prediction training objective to be a genuinely valid,
non-trivial task at all.



### Why Causal Masking Is Essential for the Training Objective
If a decoder-only model could see future tokens while predicting the current one, "predicting the next token" would become a trivial, meaningless task — the model could just directly copy the answer from a position it can already see, rather than genuinely learning to predict it from context. Causal masking (implemented via exactly the same `masked_fill` + softmax mechanism from Chapter 49, using this specific triangular pattern instead of a padding mask) is what makes next-token prediction a genuine, non-trivial supervised learning signal computable from **any** raw text with no additional labeling required — this is precisely why LLMs can be pretrained on massive, unlabeled text corpora: every sentence in the training data provides its own labels for free (each token's "true label" is simply the token that actually comes next).

## 4. Scale — Why "Large" Genuinely Matters


In [ ]:
print("""
GPT-2 (2019): ~1.5 billion parameters
GPT-3 (2020): ~175 billion parameters
Modern frontier LLMs (2023+): often 10s to 100s of billions of parameters, sometimes MORE

EMERGENT CAPABILITIES: research has repeatedly shown that certain capabilities
(few-shot learning, multi-step reasoning, following complex instructions) appear
to EMERGE somewhat suddenly at certain scale thresholds, rather than improving
purely smoothly and predictably with size -- a genuinely active, debated area of
research, not a fully settled science.

WHY THIS COURSE BUILT UP TO THIS POINT GRADUALLY: an LLM is NOT a fundamentally
different KIND of model from the SimpleTransformerEncoder built in Chapter 51 --
it's the SAME core architecture (embeddings + positional info + stacked
attention/feedforward blocks), with:
  - CAUSAL masking (this chapter) instead of bidirectional attention
  - VASTLY more parameters, layers, and training data
  - Additional engineering refinements (Chapter 57-58's fine-tuning techniques,
    RLHF and other ALIGNMENT techniques beyond this course's current scope)

Understanding THIS course's Chapters 48-54 genuinely IS understanding the core
architecture underlying GPT, LLaMA, and essentially every modern general-purpose
LLM -- scale and engineering refinement, not a fundamentally different architecture,
is what separates them from what you've already built.
""")



GPT-2 (2019): ~1.5 billion parameters
GPT-3 (2020): ~175 billion parameters
Modern frontier LLMs (2023+): often 10s to 100s of billions of parameters, sometimes MORE

EMERGENT CAPABILITIES: research has repeatedly shown that certain capabilities
(few-shot learning, multi-step reasoning, following complex instructions) appear
to EMERGE somewhat suddenly at certain scale thresholds, rather than improving
purely smoothly and predictably with size -- a genuinely active, debated area of
research, not a fully settled science.

WHY THIS COURSE BUILT UP TO THIS POINT GRADUALLY: an LLM is NOT a fundamentally
different KIND of model from the SimpleTransformerEncoder built in Chapter 51 --
it's the SAME core architecture (embeddings + positional info + stacked
attention/feedforward blocks), with:
  - CAUSAL masking (this chapter) instead of bidirectional attention
  - VASTLY more parameters, layers, and training data
  - Additional engineering refinements (Chapter 57-58's fine-tuning techniques,

### Common Mistakes
- Assuming LLMs use some fundamentally different, more exotic architecture than the Transformer this course already built — the core architecture is genuinely the same; scale and specific engineering choices (causal masking, training data, alignment techniques) are what differ.
- Confusing encoder-only (BERT-style, bidirectional) and decoder-only (GPT-style, causal) attention patterns — they solve genuinely different problems and aren't interchangeable for a given task.
- Underestimating how directly the next-token-prediction training objective connects to the causal mask — the mask isn't just an implementation detail, it's what makes the entire "free, unlabeled supervision from raw text" training paradigm valid at all.

### Interview Insight
**Q: Why is causal (triangular) masking essential for training a decoder-only language model like GPT, beyond just being an architectural choice?**
A: The entire training objective — predict the next token given everything before it — requires that the model genuinely cannot see the token it's being asked to predict (or any token after it) while making that prediction; without causal masking, self-attention would let every position freely attend to every other position, including future ones, making "predicting" the next token trivial (the model could simply copy the answer directly from a position it can already see) rather than a genuine, non-trivial predictive task learned from context. This masking is precisely what makes next-token prediction a valid, meaningful supervised signal computable directly from raw, unlabeled text — every token in any text sequence automatically provides a free training label (itself), which is exactly why LLMs can be pretrained at massive scale on raw internet text without requiring any human labeling effort for the pretraining phase itself.

### Exercise
1. Implement `causal_mask()` from memory and verify its shape/pattern for `seq_len=6`.
2. Explain, in your own words, why BERT (bidirectional) would be a poor choice for open-ended text generation, connecting this directly to the causal masking concept.

### Quiz (Chapter 54)
1. What's the key architectural difference between encoder-only and decoder-only attention?
2. Why does causal masking make next-token prediction a valid training objective from raw, unlabeled text?
3. In what specific sense is a modern LLM "the same" architecture as the Transformer built in Chapters 49-51?


---
# CHAPTER 55 — BUILD A MINI LANGUAGE MODEL

## 1. Concept
This chapter assembles Chapters 51 (Transformer encoder) and 54 (causal masking) into a **complete, working, decoder-only language model** — genuinely the same core architecture as GPT, just far smaller, trained on a small custom dataset for educational purposes rather than a massive internet-scale corpus.

## 2. The Complete Mini-GPT Architecture


In [ ]:
import torch
import torch.nn as nn
import math

class CausalSelfAttention(nn.Module):
    """Self-attention WITH causal masking baked in -- the decoder-only variant
    of Chapter 49's attention mechanism."""

    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True, dropout=dropout)

    def forward(self, x):
        seq_len = x.size(1)
        # is_causal=True tells nn.MultiheadAttention to apply causal masking INTERNALLY --
        # a convenient, efficient built-in alternative to manually constructing Chapter 54's mask
        attn_mask = nn.Transformer.generate_square_subsequent_mask(seq_len)
        output, _ = self.attn(x, x, x, attn_mask=attn_mask, is_causal=True)
        return output


class MiniGPTBlock(nn.Module):
    """A decoder block: causal self-attention + feed-forward, EXACTLY Chapter 50's
    TransformerBlock structure, but using CAUSAL attention instead of bidirectional."""

    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.causal_attn = CausalSelfAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, embed_dim),
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.norm1(x + self.dropout(self.causal_attn(x)))
        x = self.norm2(x + self.dropout(self.feed_forward(x)))
        return x


class MiniGPT(nn.Module):
    """A COMPLETE, minimal, decoder-only language model -- genuinely the same
    architectural family as GPT-2/GPT-3, at a tiny, educational scale."""

    def __init__(self, vocab_size, embed_dim=128, num_heads=8, ff_dim=512, num_layers=4, max_seq_len=256, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(max_seq_len, embed_dim)   # LEARNED positional embedding here (Ch 50's note)
        self.blocks = nn.ModuleList([MiniGPTBlock(embed_dim, num_heads, ff_dim, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)   # projects back to VOCABULARY SIZE -- a score per possible next token
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids):
        batch, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)   # (1, seq_len)

        x = self.token_embedding(token_ids) + self.position_embedding(positions)   # ADD, per Ch 50
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)   # (batch, seq_len, vocab_size) -- raw logits per position, Ch 19's rule
        return logits


model = MiniGPT(vocab_size=1000, embed_dim=128, num_heads=8, num_layers=4, max_seq_len=64)
token_ids = torch.randint(0, 1000, (4, 20))   # (batch=4, seq_len=20)
logits = model(token_ids)
print("Input shape:", token_ids.shape)
print("Output (logits) shape:", logits.shape, " (batch, seq_len, vocab_size)")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


Input shape: torch.Size([4, 20])
Output (logits) shape: torch.Size([4, 20, 1000])  (batch, seq_len, vocab_size)

Total parameters: 1,058,536


### Line-by-Line Explanation
- `lm_head = nn.Linear(embed_dim, vocab_size)` — the **language modeling head**: projects each position's `embed_dim`-sized representation into a `vocab_size`-length vector of raw logits — one score per possible next token, exactly analogous to a classification head's per-class logits (Chapter 19), just with "vocabulary size" as the class count, and computed **independently for every position** in the sequence (not just once per sample, as in Chapter 52's sequence classification).
- `nn.Transformer.generate_square_subsequent_mask(seq_len)` — a built-in PyTorch utility that directly generates exactly Chapter 54's triangular causal mask, saving you from reimplementing it manually.

## 3. Training on Next-Token Prediction


In [ ]:
def train_step_language_model(model, token_ids, criterion, optimizer):
    """The core LANGUAGE MODELING training step: predict token[i+1] from token[i] --
    genuinely just Chapter 19's CrossEntropyLoss, applied cleverly via SHIFTING."""
    inputs = token_ids[:, :-1]     # every token EXCEPT the last -- what the model SEES
    targets = token_ids[:, 1:]     # every token EXCEPT the first -- what the model must PREDICT

    logits = model(inputs)          # (batch, seq_len-1, vocab_size)

    # CrossEntropyLoss expects (N, C) logits and (N,) targets -- FLATTEN the batch AND
    # sequence dimensions together into one long list of independent predictions
    loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

sample_batch = torch.randint(0, 1000, (4, 21))   # ONE extra token, since inputs/targets each use 20 of the 21
loss = train_step_language_model(model, sample_batch, criterion, optimizer)
print("One training step's loss:", loss)


One training step's loss: 7.04088830947876


### Line-by-Line Explanation — the "Shift by One" Trick
- `inputs = token_ids[:, :-1]` / `targets = token_ids[:, 1:]` — this is the **entire** trick behind training a language model on raw, unlabeled text: given a sequence `[A, B, C, D]`, the model's input is `[A, B, C]` and its targets are `[B, C, D]` — at position 0, given `A`, it must predict `B`; at position 1, given `A, B` (via causal attention), it must predict `C`; and so on. **Every single token in the raw sequence serves as both an input (for predicting what follows) and a target (what should have been predicted at the previous position)** — this single shift operation is precisely what turns any raw text into a fully-labeled supervised training set, with zero additional annotation required, exactly Chapter 54's "free labels from raw text" point made concrete in code.
- `logits.reshape(-1, logits.size(-1))` — `CrossEntropyLoss` (Chapter 19) expects `(N, num_classes)` logits and `(N,)` targets; here, "N" needs to include **every position of every sequence in the batch** as an independent prediction — flattening `(batch, seq_len, vocab_size)` into `(batch*seq_len, vocab_size)` achieves exactly this, treating each position's next-token prediction as one independent classification instance for loss-computation purposes.

### Common Mistakes
- Forgetting the "shift by one" — feeding the model the same sequence as both input and target would let it trivially "predict" by just copying, since (without the shift) it would be asked to predict something it's literally already been given as input.
- Not flattening batch and sequence dimensions correctly before `CrossEntropyLoss` — a genuine, common shape-mismatch source when first implementing language model training.
- Using bidirectional (non-causal) attention for a generative language model — would let the model see the very tokens it's supposed to be predicting, making next-token prediction meaningless (Chapter 54's core lesson, now directly relevant to a training bug if forgotten).

### Interview Insight
**Q: Explain the "shift by one" trick used to train a language model, and why it's what makes pretraining on raw, unlabeled text possible.**
A: Given any raw sequence of tokens, you create the training input as the sequence with its last token removed, and the training target as the same sequence with its first token removed — so at every position, the model's target is simply "the token that actually came next" in the original raw text. Combined with causal masking (ensuring the model genuinely cannot see that target token while making its prediction), this transforms any piece of raw text into a fully self-labeled supervised training example, with the "labels" being nothing more than the text's own natural continuation — no human annotation is required at all. This is precisely the mechanism that makes it possible to pretrain a language model on massive volumes of raw internet text, since every sentence anywhere provides its own free supervision signal through this shift.

### Exercise
Implement `train_step_language_model` from memory, and trace through a small example sequence by hand (e.g., token IDs `[5, 12, 3, 8]`) to confirm exactly which input/target pairs the shift produces.

### Quiz (Chapter 55)
1. Walk through the "shift by one" trick with a concrete example sequence.
2. Why must `logits` and `targets` be reshaped/flattened before `CrossEntropyLoss`?
3. Why would using bidirectional attention break this entire training scheme?


---
# CHAPTER 56 — TEXT GENERATION

## 1. Concept
Once trained, a language model generates text **autoregressively**: predict one token, append it to the sequence, feed the (now longer) sequence back in, predict the next token, repeat — this chapter covers the actual generation loop and the sampling strategies controlling how the model chooses each next token from its predicted probability distribution.

## 2. The Basic Generation Loop


In [ ]:
import torch
import torch.nn.functional as F

@torch.inference_mode()   # Chapter 15's lesson -- generation is PURE inference, no gradients needed
def generate_greedy(model, start_tokens, max_new_tokens, vocab_size):
    """Greedy decoding: ALWAYS pick the single highest-probability next token."""
    model.eval()
    tokens = start_tokens.clone()

    for _ in range(max_new_tokens):
        logits = model(tokens)                      # (batch, current_seq_len, vocab_size)
        next_token_logits = logits[:, -1, :]           # only the LAST position's prediction matters
        next_token = next_token_logits.argmax(dim=-1, keepdim=True)   # Chapter 9's argmax, applied here

        tokens = torch.cat([tokens, next_token], dim=1)   # Chapter 11's cat -- APPEND the new token

    return tokens


start = torch.randint(0, 1000, (1, 5))   # a "prompt" of 5 tokens
generated = generate_greedy(model, start, max_new_tokens=10, vocab_size=1000)
print("Start tokens:", start)
print("Generated (start + 10 new tokens):", generated)
print("Generated length:", generated.shape[1], "(5 original + 10 newly generated)")


Start tokens: tensor([[343, 848, 900, 164, 130]])
Generated (start + 10 new tokens): tensor([[343, 848, 900, 164, 130, 665, 449, 561, 499, 665,  48, 364, 616, 336,
         733]])
Generated length: 15 (5 original + 10 newly generated)


### Line-by-Line Explanation
- `logits[:, -1, :]` — even though the model computes logits for **every** position in the current sequence (Chapter 55's shape), only the **last** position's prediction is actually relevant for generation — it represents "what comes after everything so far," exactly what's needed to extend the sequence by one more token.
- `torch.cat([tokens, next_token], dim=1)` — directly reuses Chapter 11's concatenation (not `stack()` — we're extending an *existing* dimension, the sequence length, not creating a new one) to grow the sequence by exactly one token each iteration.
- **Greedy decoding's limitation**: always picking the single highest-probability token produces **deterministic**, often repetitive/bland output — real language has genuine variability, and greedy decoding cannot express any of it (the same prompt always produces the exact same continuation).

## 3. Sampling Strategies — Introducing Controlled Randomness


In [ ]:
@torch.inference_mode()
def generate_with_temperature(model, start_tokens, max_new_tokens, temperature=1.0):
    """TEMPERATURE sampling: scales the logits before softmax, controlling
    how 'confident' vs 'random' the sampling distribution is."""
    model.eval()
    tokens = start_tokens.clone()

    for _ in range(max_new_tokens):
        logits = model(tokens)
        next_token_logits = logits[:, -1, :] / temperature   # THE KEY LINE

        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)   # SAMPLE from the distribution, don't just argmax

        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens


print("""
TEMPERATURE's effect on the softmax distribution:
  temperature < 1.0  ->  SHARPENS the distribution (more confident, closer to greedy/argmax)
  temperature = 1.0  ->  the model's OWN learned probability distribution, unmodified
  temperature > 1.0  ->  FLATTENS the distribution (more random, more diverse, but
                          riskier -- can produce less coherent text at HIGH temperatures)

Dividing logits by a temperature < 1 makes the LARGEST logit relatively even LARGER
after softmax (since softmax is EXPONENTIAL -- dividing by a small number amplifies
differences); dividing by temperature > 1 does the OPPOSITE, compressing differences
and flattening the resulting probability distribution.
""")

generated_temp = generate_with_temperature(model, start, max_new_tokens=10, temperature=0.7)
print("\nGenerated with temperature=0.7:", generated_temp)



TEMPERATURE's effect on the softmax distribution:
  temperature < 1.0  ->  SHARPENS the distribution (more confident, closer to greedy/argmax)
  temperature = 1.0  ->  the model's OWN learned probability distribution, unmodified
  temperature > 1.0  ->  FLATTENS the distribution (more random, more diverse, but
                          riskier -- can produce less coherent text at HIGH temperatures)

Dividing logits by a temperature < 1 makes the LARGEST logit relatively even LARGER
after softmax (since softmax is EXPONENTIAL -- dividing by a small number amplifies
differences); dividing by temperature > 1 does the OPPOSITE, compressing differences
and flattening the resulting probability distribution.


Generated with temperature=0.7: tensor([[343, 848, 900, 164, 130, 528, 791, 455, 389,   1, 756, 485, 704, 165,
         695]])


## 4. Top-k and Top-p (Nucleus) Sampling — Restricting the Candidate Pool


In [ ]:
@torch.inference_mode()
def generate_top_k(model, start_tokens, max_new_tokens, k=10, temperature=1.0):
    """TOP-K sampling: restrict sampling to only the K highest-probability tokens,
    zeroing out the probability of everything else BEFORE sampling."""
    model.eval()
    tokens = start_tokens.clone()

    for _ in range(max_new_tokens):
        logits = model(tokens)
        next_token_logits = logits[:, -1, :] / temperature

        top_k_values, top_k_indices = torch.topk(next_token_logits, k, dim=-1)
        # Set every logit OUTSIDE the top-k to -inf -- exactly Chapter 49's masking pattern,
        # here used to restrict the SAMPLING CANDIDATE POOL rather than attention
        filtered_logits = torch.full_like(next_token_logits, float("-inf"))
        filtered_logits.scatter_(-1, top_k_indices, top_k_values)

        probs = F.softmax(filtered_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens = torch.cat([tokens, next_token], dim=1)

    return tokens


generated_topk = generate_top_k(model, start, max_new_tokens=10, k=20, temperature=0.8)
print("Generated with top-k=20, temperature=0.8:", generated_topk)

print("""
WHY top-k/top-p MATTER: pure temperature sampling still allows (with SOME small
probability) sampling a genuinely bizarre, very-low-probability token -- which can
derail generation into incoherence. Top-k restricts sampling to only the k MOST
LIKELY candidates, eliminating the long, unlikely "tail" of the distribution
entirely, while STILL allowing genuine diversity among the plausible options.

Top-p (nucleus) sampling is a closely related, often-preferred alternative: instead
of a FIXED number k of candidates, it includes just enough of the highest-probability
tokens for their CUMULATIVE probability to reach a threshold p (e.g., 0.9) --
adapting the candidate pool SIZE dynamically based on how "peaked" or "flat" the
model's distribution happens to be at each specific step, rather than a fixed
count that might be too restrictive or too permissive depending on context.
""")


Generated with top-k=20, temperature=0.8: tensor([[343, 848, 900, 164, 130,  92, 206, 154, 487, 851, 220, 974, 538, 206,
         112]])

WHY top-k/top-p MATTER: pure temperature sampling still allows (with SOME small
probability) sampling a genuinely bizarre, very-low-probability token -- which can
derail generation into incoherence. Top-k restricts sampling to only the k MOST
LIKELY candidates, eliminating the long, unlikely "tail" of the distribution
entirely, while STILL allowing genuine diversity among the plausible options.

Top-p (nucleus) sampling is a closely related, often-preferred alternative: instead
of a FIXED number k of candidates, it includes just enough of the highest-probability
tokens for their CUMULATIVE probability to reach a threshold p (e.g., 0.9) --
adapting the candidate pool SIZE dynamically based on how "peaked" or "flat" the
model's distribution happens to be at each specific step, rather than a fixed
count that might be too restrictive or too permissive de

### Common Mistakes
- Using pure greedy decoding for any task where diverse, creative, or natural-sounding output matters — produces deterministic, often repetitive text.
- Setting temperature too high without top-k/top-p restriction — risks sampling genuinely incoherent, very-low-probability tokens.
- Forgetting `@torch.inference_mode()` / `model.eval()` during generation — wastes memory building an unnecessary computational graph, and (if Dropout/BatchNorm are present) introduces unwanted randomness/instability into generation.
- Not truncating/managing the growing sequence length against the model's `max_seq_len` — generation that runs long enough will eventually exceed the positional embedding's supported range, causing an error or requiring a sliding-window strategy.

### Interview Insight
**Q: What's the difference between top-k and top-p (nucleus) sampling, and why might top-p often be preferred?**
A: Top-k restricts the sampling candidate pool to a **fixed number** of the highest-probability tokens at every generation step, regardless of how the actual probability mass is distributed among them. Top-p instead includes just enough of the highest-probability tokens for their **cumulative probability** to reach a threshold `p` — this adapts the effective candidate pool size dynamically: when the model is very confident (probability mass concentrated on very few tokens), top-p naturally uses a small candidate pool; when the model is more uncertain (probability spread more evenly across many plausible tokens), top-p naturally allows a larger pool. This adaptivity is often preferred over top-k's fixed count, which can be unnecessarily restrictive in high-uncertainty situations or unnecessarily permissive in high-confidence situations.

### Exercise
1. Implement `generate_greedy` from memory and trace through what it would produce for a trivial, deterministic toy model.
2. Implement top-p (nucleus) sampling, using `torch.cumsum` on sorted, softmaxed probabilities to determine the cutoff.

### Quiz (Chapter 56)
1. Why does greedy decoding produce deterministic output, and what's its main limitation?
2. Mathematically, why does dividing logits by a temperature < 1 sharpen the resulting distribution?
3. What specific failure mode do top-k/top-p sampling both guard against that pure temperature sampling alone does not?
4. What must you always do (mode-wise) before running text generation, and why?


---
# CHAPTER 57 — LLM FINE-TUNING

## 1. Concept
Fine-tuning adapts a pretrained LLM to a specific downstream task or desired behavior — directly extending Chapter 38's transfer learning workflow, now applied to language models. This chapter covers full fine-tuning's mechanics and its genuine, serious practical challenges at LLM scale.

## 2. Full Fine-Tuning — the Direct Extension of Chapter 38


In [ ]:
import torch

print("""
FULL fine-tuning means: EVERY parameter in the pretrained model remains trainable
(requires_grad=True for ALL of them), and the ENTIRE model is updated via gradient
descent on your specific downstream task's data -- structurally IDENTICAL to
Chapter 38's "unfreeze everything" extreme, just applied to a language model
instead of a CNN.

THE GENUINE, SERIOUS CHALLENGE AT LLM SCALE: a 7-BILLION-parameter model, in
float32, requires roughly 28 GB just to STORE the weights (7B * 4 bytes).
FULL fine-tuning ALSO requires storing:
  - Gradients for EVERY parameter (another ~28 GB)
  - Optimizer state (Adam/AdamW keeps TWO extra values PER PARAMETER --
    the first and second moment estimates, Chapter 20 -- another ~56 GB)

TOTAL: potentially 100+ GB of memory just for a 7B model's full fine-tuning --
FAR beyond what even a high-end SINGLE consumer/prosumer GPU provides (typically
24-80 GB), genuinely requiring multiple GPUs or specialized memory-optimization
techniques (Chapter 75's gradient checkpointing, Chapter 78's distributed training)
even for a model this "small" by modern LLM standards.
""")



FULL fine-tuning means: EVERY parameter in the pretrained model remains trainable
(requires_grad=True for ALL of them), and the ENTIRE model is updated via gradient
descent on your specific downstream task's data -- structurally IDENTICAL to
Chapter 38's "unfreeze everything" extreme, just applied to a language model
instead of a CNN.

THE GENUINE, SERIOUS CHALLENGE AT LLM SCALE: a 7-BILLION-parameter model, in
float32, requires roughly 28 GB just to STORE the weights (7B * 4 bytes).
FULL fine-tuning ALSO requires storing:
  - Gradients for EVERY parameter (another ~28 GB)
  - Optimizer state (Adam/AdamW keeps TWO extra values PER PARAMETER --
    the first and second moment estimates, Chapter 20 -- another ~56 GB)

TOTAL: potentially 100+ GB of memory just for a 7B model's full fine-tuning --
FAR beyond what even a high-end SINGLE consumer/prosumer GPU provides (typically
24-80 GB), genuinely requiring multiple GPUs or specialized memory-optimization
techniques (Chapter 75's gradient

## 3. Warmup, Learning Rate, and Catastrophic Forgetting — At LLM Scale
Directly extending Chapter 21's warmup discussion and Chapter 38's catastrophic forgetting concern, both matter **even more** at LLM scale:


In [ ]:
print("""
LLM fine-tuning typically uses:
  - VERY SMALL learning rates (often 1e-5 to 1e-6, notably smaller than the 1e-3 to
    1e-4 range common for training a smaller model from scratch, Ch 20-22) --
    the pretrained model's weights represent a HUGE amount of learned general
    knowledge; large updates risk destroying it FAST (catastrophic forgetting,
    Ch 38, now a MUCH more consequential risk given how much more the model
    "knows" before fine-tuning even begins).

  - SHORT warmup periods, then a gentle decay (Ch 21) -- the same warmup rationale
    from Ch 21 applies, now specifically protecting a model that already
    represents an enormous pretraining investment from early, large, disruptive
    updates.

  - Often just a FEW EPOCHS (sometimes even LESS than one full pass over the
    fine-tuning dataset) -- unlike training from scratch, which typically needs
    MANY epochs to converge, fine-tuning an already-capable pretrained model
    toward a NEW but related task/behavior often needs comparatively little
    additional training to adapt successfully -- MORE fine-tuning is not always
    better, and can itself risk overfitting to a (often much smaller) fine-tuning
    dataset, or DEGRADING the model's broader general capabilities.
""")



LLM fine-tuning typically uses:
  - VERY SMALL learning rates (often 1e-5 to 1e-6, notably smaller than the 1e-3 to
    1e-4 range common for training a smaller model from scratch, Ch 20-22) --
    the pretrained model's weights represent a HUGE amount of learned general
    knowledge; large updates risk destroying it FAST (catastrophic forgetting,
    Ch 38, now a MUCH more consequential risk given how much more the model
    "knows" before fine-tuning even begins).

  - SHORT warmup periods, then a gentle decay (Ch 21) -- the same warmup rationale
    from Ch 21 applies, now specifically protecting a model that already
    represents an enormous pretraining investment from early, large, disruptive
    updates.

  - Often just a FEW EPOCHS (sometimes even LESS than one full pass over the
    fine-tuning dataset) -- unlike training from scratch, which typically needs
    MANY epochs to converge, fine-tuning an already-capable pretrained model
    toward a NEW but related task/behavior

### Common Mistakes
- Using a learning rate borrowed from from-scratch training for LLM fine-tuning — a genuinely serious mistake, risking rapid catastrophic forgetting of the pretrained model's valuable general capabilities.
- Fine-tuning for too many epochs on a small task-specific dataset — can cause the model to overfit narrowly to that dataset, degrading its broader general capabilities (a specific, LLM-relevant instance of Chapter 31's overfitting concern) — sometimes called "alignment tax" or capability regression in this specific context.
- Attempting full fine-tuning of a large model without first verifying available memory can actually support it — a very common, expensive mistake; Chapter 58's PEFT techniques exist specifically to make fine-tuning tractable without this memory burden.
- Not evaluating the fine-tuned model on BOTH the target task AND a broader capability benchmark — a model that improves substantially on the fine-tuning task while silently regressing on general capabilities is a genuine risk worth actively checking for, not simply assuming away.

### Interview Insight
**Q: Why does full fine-tuning of a large language model require dramatically more memory than the model's own parameter count alone would suggest, and what does this practically mean for who can do it?**
A: Full fine-tuning requires storing not just the model's weights, but also a gradient for every single trainable parameter (roughly doubling the memory just for weights + gradients), plus — for adaptive optimizers like Adam/AdamW (Chapter 20) — two additional per-parameter state values (first and second moment estimates), which can roughly triple the total memory requirement beyond the weights alone. For a model with billions of parameters, this compounds into memory requirements far exceeding what even high-end single GPUs typically provide, genuinely requiring either multiple GPUs, specialized memory-optimization techniques, or — as Chapter 58 covers — parameter-efficient fine-tuning methods that dramatically reduce how many parameters actually need gradients and optimizer state in the first place, making fine-tuning large models tractable on much more modest hardware.

### Exercise
Compute, by hand, the approximate total memory requirement (weights + gradients + AdamW optimizer state, all in float32) for full fine-tuning a 1-billion-parameter model, and compare it to a typical consumer GPU's memory (e.g., 24 GB).

### Quiz (Chapter 57)
1. Why does AdamW's optimizer state roughly triple (not just double) the memory needed beyond the model's weights alone during full fine-tuning?
2. Why does LLM fine-tuning typically use a much smaller learning rate than from-scratch training?
3. What risk does fine-tuning for too many epochs on a small dataset pose, specifically for a large pretrained model?


---
# CHAPTER 58 — LORA AND PEFT

## 1. Concept
**PEFT** (Parameter-Efficient Fine-Tuning) is a family of techniques that adapt a large pretrained model to a new task by training only a **small fraction** of its parameters (often less than 1%), directly solving Chapter 57's memory problem — **LoRA** (Low-Rank Adaptation) is the most widely-used PEFT technique in current practice.

## 2. LoRA's Core Idea
Instead of directly updating a large pretrained weight matrix `W` (shape `d × d`, potentially enormous), LoRA **freezes `W` entirely** and instead learns a small **update** expressed as the product of two much smaller matrices:
```
W_new = W_frozen + (A @ B) * scaling
```
where `A` has shape `(d, r)` and `B` has shape `(r, d)`, with `r` (the "rank") chosen to be **much smaller** than `d` (commonly `r=8` or `r=16`, versus `d` potentially in the thousands) — the number of trainable parameters in `A` and `B` combined (`2 * d * r`) is dramatically smaller than `W`'s own `d * d` parameters, especially as `d` grows large.

## 3. Implementing a LoRA Layer From Scratch


In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    """Wraps an EXISTING, FROZEN nn.Linear layer, adding a small, trainable
    low-rank update alongside it -- the core LoRA mechanism, implemented directly."""

    def __init__(self, original_linear: nn.Linear, rank=8, alpha=16):
        super().__init__()
        self.original_linear = original_linear
        for param in self.original_linear.parameters():
            param.requires_grad = False   # FREEZE the original weight -- Chapter 38's pattern, applied here

        in_features = original_linear.in_features
        out_features = original_linear.out_features

        # A and B are the SMALL, TRAINABLE low-rank matrices
        self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)   # small random init
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))          # ZERO init -- IMPORTANT (see below)

        self.scaling = alpha / rank
        self.rank = rank

    def forward(self, x):
        original_output = self.original_linear(x)          # the FROZEN pretrained computation
        lora_update = (x @ self.lora_A @ self.lora_B) * self.scaling   # the SMALL, trainable addition
        return original_output + lora_update                 # combined, exactly W_new's formula above


original_layer = nn.Linear(768, 768)   # a plausible Transformer hidden-layer size
lora_layer = LoRALinear(original_layer, rank=8, alpha=16)

x = torch.randn(4, 10, 768)
output = lora_layer(x)
print("Output shape:", output.shape, " (identical to the ORIGINAL layer's output shape)")

trainable = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in lora_layer.parameters() if not p.requires_grad)
print(f"\nTrainable parameters (LoRA A + B only): {trainable:,}")
print(f"Frozen parameters (original weight): {frozen:,}")
print(f"Trainable fraction: {trainable / (trainable + frozen):.2%}")


Output shape: torch.Size([4, 10, 768])  (identical to the ORIGINAL layer's output shape)

Trainable parameters (LoRA A + B only): 12,288
Frozen parameters (original weight): 590,592
Trainable fraction: 2.04%


### Line-by-Line Explanation — Why Each Design Choice Matters
- `self.lora_B = nn.Parameter(torch.zeros(rank, out_features))` — **zero-initializing `B`** is a deliberate, important choice: since `lora_update = x @ A @ B`, and `B` starts at exactly zero, the LoRA update contributes **exactly zero** at the very start of training — meaning the model's behavior is **identical to the original pretrained model** before any fine-tuning steps have happened, and training gradually, smoothly introduces the adaptation from a genuinely neutral starting point, rather than immediately perturbing the model's behavior with a random, untrained update.
- `self.scaling = alpha / rank` — a scaling factor controlling how strongly the LoRA update influences the output relative to the original frozen computation; `alpha` and `rank` are both tunable hyperparameters, and their ratio is what's typically tuned in practice (many practitioners keep this ratio in a similar range even when adjusting `rank` itself).
- The trainable parameter count comparison at the end makes PEFT's core value proposition concrete: for a `768×768` layer, LoRA with `rank=8` trains roughly `2 × 768 × 8 ≈ 12,288` parameters instead of the original `768×768 ≈ 589,824` — **under 2%** of the original layer's parameter count, and this ratio becomes even more dramatic for the much larger weight matrices found in actual large-scale LLMs.

## 4. In Practice — the `peft` Library


In [ ]:
# In real practice, you'd use Hugging Face's dedicated `peft` library rather than
# manually wrapping every layer as done above (illustrative -- requires internet
# access this sandbox doesn't have)

print("""
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                                    # the RANK -- this chapter's core hyperparameter
    lora_alpha=16,                           # the SCALING numerator
    target_modules=["q_proj", "v_proj"],       # WHICH layers to apply LoRA to -- typically
                                                 # attention's query/value projections specifically,
                                                 # empirically found to be highly effective targets
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Output resembles: "trainable params: 4,194,304 || all params: 6,738,415,616 || trainable%: 0.0622"
""")

print("""
This ~0.06% trainable-parameter figure is GENUINELY REPRESENTATIVE of real LoRA
fine-tuning runs on large models -- and directly explains why LoRA makes it
possible to fine-tune multi-billion-parameter models on a SINGLE consumer GPU,
something Chapter 57 established full fine-tuning simply cannot do at that scale.

ADDITIONAL PEFT BENEFITS beyond memory:
  - The frozen base model's weights can be SHARED across many different LoRA
    "adapters" (one small adapter per task/customer/use case) -- storing many
    task-specific adaptations costs FAR less than storing many full fine-tuned
    COPIES of the entire base model.
  - LoRA adapters can be MERGED back into the base weights after training
    (W_frozen + A@B*scaling, computed ONCE) for zero additional inference
    latency, OR kept separate for flexible swapping between multiple adapters
    at inference time.
""")



from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,                                    # the RANK -- this chapter's core hyperparameter
    lora_alpha=16,                           # the SCALING numerator
    target_modules=["q_proj", "v_proj"],       # WHICH layers to apply LoRA to -- typically
                                                 # attention's query/value projections specifically,
                                                 # empirically found to be highly effective targets
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Output resembles: "trainable params: 4,194,304 || all params: 6,738,415,616 || trainable%: 0.0622"


This ~0.06% trainable-parameter figure is GENUINELY REPRESENTATIVE of real LoRA
fine-tuning runs on large models -- and directly explains why LoRA makes it
possible to fine-tune multi-billion-parameter models on a SINGLE consumer GPU,
something

### Common Mistakes
- Randomly initializing **both** `A` and `B` (rather than zero-initializing `B` specifically) — breaks LoRA's "starts identical to the base model" property, causing an immediate, untrained perturbation to the model's behavior right at the start of fine-tuning.
- Choosing `rank` too small for a task requiring substantial adaptation — LoRA's expressiveness is fundamentally limited by its rank; too small a rank may underfit the target task, exactly the Chapter 31 bias-variance-style tradeoff, now applied to the adaptation's own capacity rather than the base model's.
- Applying LoRA to every single layer/parameter indiscriminately rather than targeting specific, empirically-effective layers (commonly attention projections) — increases trainable parameter count and training cost without necessarily improving adaptation quality proportionally.
- Forgetting to actually freeze the original weights (`requires_grad=False`) when implementing LoRA manually — without this, you're not actually getting PEFT's memory benefit at all, since the (much larger) original weights would still require gradients and optimizer state.

### Interview Insight
**Q: Why is `lora_B` specifically initialized to all zeros, while `lora_A` is initialized randomly, in a standard LoRA implementation?**
A: The LoRA update to the layer's output is computed as `x @ A @ B`, scaled — with `B` initialized to exactly zero, this entire update evaluates to zero at the very start of training, regardless of what `A` contains, meaning the LoRA-augmented model's behavior is initially **identical** to the original frozen pretrained model. This gives fine-tuning a clean, neutral starting point — training then gradually and smoothly introduces the task-specific adaptation as `B`'s values move away from zero, rather than immediately injecting a random, untrained perturbation into the model's behavior from the very first forward pass, which would happen if both `A` and `B` started randomly initialized.

### Exercise
1. Implement `LoRALinear` from memory, verify that at initialization (before any training), its output is nearly identical to the original frozen layer's output alone (since the LoRA update starts at zero).
2. Compute the trainable-parameter percentage for LoRA with `rank=16` applied to a `4096×4096` weight matrix (a plausible size in a larger LLM), and compare it to full fine-tuning of that same matrix.

### Quiz (Chapter 58)
1. Write out LoRA's core update formula from memory: `W_new = ...`
2. Why is zero-initializing `B` (not `A`) specifically important?
3. What's the practical benefit of being able to "merge" a LoRA adapter back into the base weights after training, versus keeping it separate?
4. Why does training on the order of 0.06% of a model's total parameters still meaningfully adapt its behavior?

### Chapter 58 Summary
- PEFT techniques, especially LoRA, solve Chapter 57's full-fine-tuning memory problem by training only a small, low-rank update alongside a frozen pretrained model — often under 1% of total parameters.
- LoRA's zero-initialized `B` matrix ensures training starts from behavior identical to the base model, adapting gradually and smoothly from there.
- This is precisely the technique that makes fine-tuning multi-billion-parameter LLMs tractable on a single consumer GPU, a capability full fine-tuning (Chapter 57) simply cannot provide at that scale.

---

## END OF NOTEBOOK 10 — What You Should Now Be Able To Do
- Explain precisely why Hugging Face's `transformers` library requires no new fundamental knowledge beyond what this course has already built — every model is a genuine `nn.Module`.
- Explain the encoder-only/decoder-only/encoder-decoder architectural family tree, and implement causal masking, connecting it directly to why next-token prediction is a valid training objective on raw text.
- Build a complete, working mini-GPT-style language model from scratch, including the "shift by one" training trick.
- Implement autoregressive text generation with greedy decoding, temperature sampling, and top-k sampling, explaining each strategy's tradeoffs.
- Explain the memory challenges of full LLM fine-tuning precisely (weights + gradients + optimizer state) and why learning rate/epoch choices differ from from-scratch training.
- Implement LoRA from scratch, explain its zero-initialization design choice, and articulate why PEFT techniques make large-model fine-tuning tractable on consumer hardware.

## Before Moving to Notebook 11 (Chapters 59–64: Model Quantization, ONNX and Model Export, TorchScript, Model Serving, REST API Deployment, Docker for PyTorch)
 